In [2]:
import pandas as pd
import numpy as np
import csv
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2005_2006"
    / "Questionnnair"
)

xpt_file = DATA_DIR / "HUQ_D.xpt"
csv_file = DATA_DIR / "HUQ_D_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

# Untouched source copy for final fidelity checking
source_df = df.copy()


print("=" * 80)
print("NHANES 2005-2006 HUQ_D XPT -> CSV")
print("=" * 80)

print("Rows:", f"{len(df):,}")
print("Columns:", len(df.columns))


# ============================================================
# 3. EXPECTED STRUCTURE
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "HUQ010",
    "HUQ020",
    "HUQ030",
    "HUQ040",
    "HUQ050",
    "HUQ060",
    "HUQ071",
    "HUD080",
    "HUQ090"
]


if len(df) != 10348:
    raise ValueError(
        f"Unexpected row count: {len(df):,}"
    )


if len(df.columns) != 10:
    raise ValueError(
        f"Unexpected column count: {len(df.columns)}"
    )


if df.columns.tolist() != EXPECTED_COLUMNS:
    raise ValueError(
        "HUQ_D columns/order differ.\n"
        f"Actual:\n{df.columns.tolist()}"
    )


if df.columns.duplicated().any():
    raise ValueError(
        "Duplicate column names found."
    )


print(
    "PASS: HUQ_D structure = "
    "10,348 rows x 10 columns."
)


# ============================================================
# 4. KNOWN SAS/XPORT TINY-ZERO ARTIFACT
# ============================================================

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


numeric_cols = df.select_dtypes(
    include=[np.number]
).columns


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)


true_zero_counts = (
    df[numeric_cols]
    .eq(0)
    .sum()
)


print("\n--- ORIGINAL XPT ZERO CHECK ---")

print(
    "True numeric zeros:",
    f"{int(true_zero_counts.sum()):,}"
)

print(
    "Tiny artifact occurrences:",
    f"{int(tiny_counts.sum()):,}"
)


print("\nTiny values by variable:")

print(
    tiny_counts[
        tiny_counts > 0
    ]
    .sort_values(
        ascending=False
    )
    .to_string()
)


# ============================================================
# 5. EXPECTED VALIDATED ZERO ARTIFACT
# ============================================================

EXPECTED_TINY_COUNTS = {
    "HUQ050": 1526
}


EXPECTED_TINY_TOTAL = 1526


# ============================================================
# 6. VALIDATE TINY ARTIFACT LOCATION
# ============================================================

actual_tiny_columns = set(
    tiny_counts[
        tiny_counts > 0
    ].index
)


expected_tiny_columns = set(
    EXPECTED_TINY_COUNTS.keys()
)


unexpected_columns = (
    actual_tiny_columns
    -
    expected_tiny_columns
)


if unexpected_columns:

    raise ValueError(
        "STOP: Tiny values found in unexpected "
        "HUQ_D variables:\n"
        + str(sorted(unexpected_columns))
    )


missing_expected_columns = (
    expected_tiny_columns
    -
    actual_tiny_columns
)


if missing_expected_columns:

    raise ValueError(
        "STOP: Expected HUQ050 tiny values missing."
    )


if int(
    tiny_counts["HUQ050"]
) != 1526:

    raise ValueError(
        "HUQ050 tiny count differs from expected 1,526."
    )


if int(
    tiny_counts.sum()
) != EXPECTED_TINY_TOTAL:

    raise ValueError(
        "Unexpected total tiny-value count."
    )


print(
    "\nPASS: Exactly 1,526 tiny values found "
    "in HUQ050 only."
)


# ============================================================
# 7. RESTORE VALIDATED HUQ050 ZEROS
# ============================================================

mask = (
    df["HUQ050"]
    .eq(TINY_VALUE)
)


print(
    "\nHUQ050 tiny -> zero:",
    int(mask.sum())
)


df.loc[
    mask,
    "HUQ050"
] = 0.0


# ============================================================
# 8. VERIFY HUQ050 ZERO RESTORATION
# ============================================================

huq050_zero_count = int(
    df["HUQ050"]
    .eq(0)
    .sum()
)


print(
    "HUQ050 zeros after restoration:",
    huq050_zero_count
)


if huq050_zero_count != 1526:

    raise ValueError(
        "HUQ050 zero restoration failed."
    )


print(
    "PASS: 1,526 HUQ050 zeros restored."
)


# ============================================================
# 9. VERIFY NO TINY ARTIFACT REMAINS
# ============================================================

remaining_tiny = int(
    df.select_dtypes(
        include=[np.number]
    )
    .eq(TINY_VALUE)
    .sum()
    .sum()
)


print(
    "\nRemaining tiny artifacts:",
    remaining_tiny
)


if remaining_tiny != 0:

    raise ValueError(
        "Tiny-value artifact remains."
    )


print(
    "PASS: No tiny-value artifacts remain."
)


# ============================================================
# 10. CONFIRM NO GENUINE DECIMAL VARIABLES
# ============================================================

print("\n--- FRACTIONAL VALUE CHECK ---")


fractional_failures = []


for col in numeric_cols:

    values = (
        df[col]
        .dropna()
        .to_numpy()
    )


    if len(values) == 0:
        continue


    fractional = ~np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )


    count = int(
        fractional.sum()
    )


    print(
        f"{col:<10}: "
        f"fractional values = {count:,}"
    )


    if count != 0:

        fractional_failures.append(
            col
        )


if fractional_failures:

    raise ValueError(
        "STOP: Genuine decimal values found in:\n"
        + str(fractional_failures)
    )


print(
    "PASS: All HUQ_D values are "
    "whole-number identifiers/codes."
)


# ============================================================
# 11. SEQN VALIDATION
# ============================================================

print("\n--- SEQN CHECK ---")


seqn_values = (
    df["SEQN"]
    .dropna()
)


if not np.all(
    np.isclose(
        seqn_values,
        np.round(seqn_values),
        rtol=0,
        atol=1e-12
    )
):

    raise ValueError(
        "SEQN contains fractional values."
    )


seqn_missing = int(
    df["SEQN"]
    .isna()
    .sum()
)


seqn_duplicates = int(
    df["SEQN"]
    .duplicated()
    .sum()
)


seqn_min = df["SEQN"].min()
seqn_max = df["SEQN"].max()


print(
    "Missing:",
    seqn_missing
)

print(
    "Duplicates:",
    seqn_duplicates
)

print(
    "Minimum:",
    seqn_min
)

print(
    "Maximum:",
    seqn_max
)


if seqn_missing != 0:
    raise ValueError(
        "Unexpected missing SEQN."
    )


if seqn_duplicates != 0:
    raise ValueError(
        "Duplicate SEQN values found."
    )


if seqn_min != 31127:
    raise ValueError(
        f"Unexpected minimum SEQN: {seqn_min}"
    )


if seqn_max != 41474:
    raise ValueError(
        f"Unexpected maximum SEQN: {seqn_max}"
    )


print(
    "PASS: SEQN validated."
)


# ============================================================
# 12. CDC EXPECTED FREQUENCIES
# ============================================================

#
# These frequencies represent the data AFTER
# the validated HUQ050 tiny -> 0 restoration.
#

EXPECTED_FREQUENCIES = {

    "HUQ010": {
        "counts": {
            1: 3143,
            2: 2694,
            3: 3066,
            4: 1204,
            5: 236,
            9: 5
        },
        "missing": 0
    },

    "HUQ020": {
        "counts": {
            1: 1923,
            2: 783,
            3: 7109,
            9: 7
        },
        "missing": 526
    },

    "HUQ030": {
        "counts": {
            1: 9027,
            2: 1274,
            3: 46,
            9: 1
        },
        "missing": 0
    },

    "HUQ040": {
        "counts": {
            1: 2309,
            2: 6304,
            3: 188,
            4: 170,
            5: 99,
            7: 1,
            9: 2
        },
        "missing": 1275
    },

    "HUQ050": {
        "counts": {
            0: 1526,
            1: 2117,
            2: 3066,
            3: 2433,
            4: 595,
            5: 602,
            99: 9
        },
        "missing": 0
    },

    "HUQ060": {
        "counts": {
            1: 68,
            2: 94,
            3: 930,
            4: 364,
            5: 67,
            9: 3
        },
        "missing": 8822
    },

    "HUQ071": {
        "counts": {
            1: 984,
            2: 9358,
            9: 6
        },
        "missing": 0
    },

    "HUD080": {
        "counts": {
            1: 736,
            2: 147,
            3: 53,
            4: 17,
            5: 12,
            6: 18,
            99999: 1
        },
        "missing": 9364
    },

    "HUQ090": {
        "counts": {
            1: 652,
            2: 8229,
            7: 1,
            9: 7
        },
        "missing": 1459
    }
}


# ============================================================
# 13. VALIDATE ALL CDC FREQUENCIES
# ============================================================

print("\n--- CDC FREQUENCY CHECKS ---")


for col, spec in EXPECTED_FREQUENCIES.items():

    actual_counts = (
        df[col]
        .value_counts(
            dropna=True
        )
        .sort_index()
        .to_dict()
    )


    actual_counts = {
        int(k): int(v)
        for k, v
        in actual_counts.items()
    }


    actual_missing = int(
        df[col]
        .isna()
        .sum()
    )


    if actual_counts != spec["counts"]:

        raise ValueError(
            f"{col}: CDC frequency mismatch.\n"
            f"Expected: {spec['counts']}\n"
            f"Actual:   {actual_counts}"
        )


    if actual_missing != spec["missing"]:

        raise ValueError(
            f"{col}: missing count mismatch.\n"
            f"Expected: {spec['missing']:,}\n"
            f"Actual:   {actual_missing:,}"
        )


    print(
        f"PASS: {col:<8} "
        f"valid={sum(actual_counts.values()):,} "
        f"missing={actual_missing:,}"
    )


print(
    "PASS: All HUQ_D frequencies match CDC."
)


# ============================================================
# 14. EXPECTED SOURCE FOR FIDELITY
# ============================================================

#
# expected_df =
#
# original XPT
# +
# ONLY:
#
# HUQ050 tiny -> 0
#

expected_df = source_df.copy()


expected_df.loc[
    expected_df["HUQ050"]
    .eq(TINY_VALUE),
    "HUQ050"
] = 0.0


# ============================================================
# 15. CLEAN SEQN + QUESTIONNAIRE FORMATTING
# ============================================================

#
# Every nonmissing value was verified to be
# an exact whole number.
#
# Therefore:
#
# SEQN
# 31127.0 -> 31127
#
# HUQ010
# 1.0 -> 1
#
# HUQ050
# 0.0 -> 0
#
# HUD080
# 99999.0 -> 99999
#
# Missing
# NaN -> blank
#

for col in df.columns:

    df[col] = (
        df[col]
        .astype("Int64")
    )


print(
    "\nPASS: SEQN and questionnaire fields "
    "converted to clean integer formatting."
)


# ============================================================
# 16. VALUE FIDELITY BEFORE EXPORT
# ============================================================

print("\n--- VALUE FIDELITY BEFORE EXPORT ---")


pre_export_failures = []


for col in expected_df.columns:

    expected_values = (
        expected_df[col]
        .to_numpy()
    )


    current_values = (
        df[col]
        .astype("Float64")
        .to_numpy(
            dtype=float,
            na_value=np.nan
        )
    )


    if not np.allclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    ):

        pre_export_failures.append(
            col
        )


if pre_export_failures:

    raise ValueError(
        "Unexpected value changes before export:\n"
        + str(pre_export_failures)
    )


print(
    "PASS: Values match expected XPT "
    "after ONLY HUQ050 zero restoration."
)


# ============================================================
# 17. EXPORT CSV
# ============================================================

#
# NO:
#
# float_format
# round()
# recalculation
#
# Missing SAS values -> blank CSV cells.
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print("\nCSV created:")
print(csv_file)


# ============================================================
# 18. RE-READ CSV WITH ROUND-TRIP PARSING
# ============================================================

csv_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


# ============================================================
# 19. STRUCTURE AFTER EXPORT
# ============================================================

if csv_df.shape != expected_df.shape:

    raise ValueError(
        "CSV shape differs from XPT.\n"
        f"Expected: {expected_df.shape}\n"
        f"Actual:   {csv_df.shape}"
    )


if csv_df.columns.tolist() != EXPECTED_COLUMNS:

    raise ValueError(
        "CSV columns/order changed."
    )


print(
    "\nPASS: CSV structure matches XPT."
)


# ============================================================
# 20. EXACT XPT -> CSV NUMERIC FIDELITY
# ============================================================

print("\n--- XPT -> CSV NUMERIC FIDELITY ---")


numeric_failures = []
total_numeric_differences = 0


for col in expected_df.columns:

    expected_values = (
        expected_df[col]
        .to_numpy()
    )


    exported_values = (
        csv_df[col]
        .to_numpy()
    )


    equal = np.isclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    differences = int(
        (~equal).sum()
    )


    total_numeric_differences += (
        differences
    )


    if differences > 0:

        numeric_failures.append(
            (
                col,
                differences
            )
        )


print(
    "Total numeric differences:",
    total_numeric_differences
)


if numeric_failures:

    raise ValueError(
        "XPT -> CSV numeric differences found:\n"
        + str(numeric_failures)
    )


print(
    "PASS: All 10 CSV columns numerically "
    "match expected XPT values."
)


# ============================================================
# 21. MISSING-POSITION FIDELITY
# ============================================================

missing_mismatches = 0
missing_failures = []


for col in expected_df.columns:

    expected_missing = (
        expected_df[col]
        .isna()
        .to_numpy()
    )


    exported_missing = (
        csv_df[col]
        .isna()
        .to_numpy()
    )


    diff = int(
        np.sum(
            expected_missing
            !=
            exported_missing
        )
    )


    missing_mismatches += diff


    if diff > 0:

        missing_failures.append(
            (
                col,
                diff
            )
        )


print(
    "\nMissing-position differences:",
    missing_mismatches
)


if missing_failures:

    raise ValueError(
        "Missing-value positions changed:\n"
        + str(missing_failures)
    )


print(
    "PASS: Missing positions preserved exactly."
)


# ============================================================
# 22. READ RAW CSV TOKENS
# ============================================================

with open(
    csv_file,
    "r",
    encoding="utf-8",
    newline=""
) as f:

    reader = csv.DictReader(f)
    raw_rows = list(reader)


if len(raw_rows) != 10348:

    raise ValueError(
        "Unexpected raw CSV row count."
    )


# ============================================================
# 23. STRICT RAW FORMAT FIDELITY
# ============================================================

#
# Every HUQ_D field is an exact whole-number
# identifier/questionnaire code.
#
# Expected:
#
# ""              if missing
# str(int(value)) otherwise
#

raw_mismatches = 0
raw_examples = []


for i, row in enumerate(
    raw_rows
):

    for col in expected_df.columns:

        source_value = (
            expected_df.iloc[i][col]
        )


        expected_text = (
            ""
            if pd.isna(source_value)
            else str(
                int(source_value)
            )
        )


        actual_text = (
            row[col]
        )


        if actual_text != expected_text:

            raw_mismatches += 1


            if len(
                raw_examples
            ) < 20:

                raw_examples.append(
                    {
                        "csv_row": i + 2,
                        "column": col,
                        "expected": expected_text,
                        "actual": actual_text
                    }
                )


print(
    "\nRaw-format differences:",
    raw_mismatches
)


if raw_mismatches != 0:

    print(
        "Examples:"
    )


    for example in raw_examples:

        print(
            example
        )


    raise ValueError(
        "HUQ_D raw CSV formatting differs."
    )


print(
    "PASS: Raw CSV formatting matches expected "
    "whole-number questionnaire representation."
)


# ============================================================
# 24. UNWANTED .0 CHECK
# ============================================================

dot_zero_count = 0
dot_zero_examples = []


for row_number, row in enumerate(
    raw_rows,
    start=2
):

    for col, value in row.items():

        if (
            value != ""
            and
            value.endswith(".0")
        ):

            dot_zero_count += 1


            if len(
                dot_zero_examples
            ) < 20:

                dot_zero_examples.append(
                    (
                        row_number,
                        col,
                        value
                    )
                )


print(
    "\nValues ending '.0':",
    dot_zero_count
)


if dot_zero_count != 0:

    print(
        "Examples:",
        dot_zero_examples
    )

    raise ValueError(
        "Unnecessary .0 formatting remains."
    )


print(
    "PASS: No unnecessary .0 values."
)


# ============================================================
# 25. SCIENTIFIC NOTATION CHECK
# ============================================================

scientific_count = 0
scientific_examples = []


for row_number, row in enumerate(
    raw_rows,
    start=2
):

    for col, value in row.items():

        lower = (
            value.lower()
        )


        if (
            "e-" in lower
            or
            "e+" in lower
        ):

            scientific_count += 1


            if len(
                scientific_examples
            ) < 20:

                scientific_examples.append(
                    (
                        row_number,
                        col,
                        value
                    )
                )


print(
    "Scientific notation occurrences:",
    scientific_count
)


if scientific_count != 0:

    print(
        "Examples:",
        scientific_examples
    )

    raise ValueError(
        "Unexpected scientific notation found."
    )


print(
    "PASS: No scientific notation."
)


# ============================================================
# 26. FINAL TINY-ARTIFACT TEXT CHECK
# ============================================================

with open(
    csv_file,
    "r",
    encoding="utf-8"
) as f:

    csv_text = f.read()


tiny_string_count = (
    csv_text.count(
        "5.397605346934028e-79"
    )
)


print(
    "Tiny artifact strings in CSV:",
    tiny_string_count
)


if tiny_string_count != 0:

    raise ValueError(
        "Tiny XPORT artifact found in CSV."
    )


print(
    "PASS: No tiny artifact in CSV."
)


# ============================================================
# 27. FINAL HUQ050 ZERO CHECK
# ============================================================

final_huq050_zeros = int(
    csv_df["HUQ050"]
    .eq(0)
    .sum()
)


print(
    "\nHUQ050 zeros in final CSV:",
    final_huq050_zeros
)


if final_huq050_zeros != 1526:

    raise ValueError(
        "Final HUQ050 zero count differs."
    )


print(
    "PASS: Final HUQ050 zeros = 1,526."
)


# ============================================================
# 28. FINAL CDC FREQUENCY CHECK
# ============================================================

print("\n--- FINAL CSV FREQUENCY CHECK ---")


for col, spec in EXPECTED_FREQUENCIES.items():

    actual_counts = (
        csv_df[col]
        .value_counts(
            dropna=True
        )
        .sort_index()
        .to_dict()
    )


    actual_counts = {
        int(k): int(v)
        for k, v
        in actual_counts.items()
    }


    actual_missing = int(
        csv_df[col]
        .isna()
        .sum()
    )


    if actual_counts != spec["counts"]:

        raise ValueError(
            f"{col}: CSV frequencies changed."
        )


    if actual_missing != spec["missing"]:

        raise ValueError(
            f"{col}: CSV missing count changed."
        )


print(
    "PASS: Final CSV frequencies still match CDC."
)


# ============================================================
# 29. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 80)
print("HUQ_D CONVERSION COMPLETE")
print("=" * 80)

print(
    "Rows:",
    f"{len(df):,}"
)

print(
    "Columns:",
    len(df.columns)
)

print(
    "Tiny artifacts in original XPT:",
    1526
)

print(
    "HUQ050 tiny -> zero:",
    1526
)

print(
    "Unexpected tiny variables:",
    0
)

print(
    "Genuine decimal variables:",
    0
)

print(
    "SEQN/code formatting:",
    "integer"
)

print(
    "XPT -> CSV numeric differences:",
    total_numeric_differences
)

print(
    "Missing-position differences:",
    missing_mismatches
)

print(
    "Raw-format differences:",
    raw_mismatches
)

print(
    "Values ending .0:",
    dot_zero_count
)

print(
    "Scientific notation:",
    scientific_count
)

print(
    "\nEXPECTED FINAL STATUS:"
)

print(
    "HUQ_D_corrected.csv = "
    "CORRECTED + FIDELITY CHECKED"
)

NHANES 2005-2006 HUQ_D XPT -> CSV
Rows: 10,348
Columns: 10
PASS: HUQ_D structure = 10,348 rows x 10 columns.

--- ORIGINAL XPT ZERO CHECK ---
True numeric zeros: 0
Tiny artifact occurrences: 1,526

Tiny values by variable:
HUQ050    1526

PASS: Exactly 1,526 tiny values found in HUQ050 only.

HUQ050 tiny -> zero: 1526
HUQ050 zeros after restoration: 1526
PASS: 1,526 HUQ050 zeros restored.

Remaining tiny artifacts: 0
PASS: No tiny-value artifacts remain.

--- FRACTIONAL VALUE CHECK ---
SEQN      : fractional values = 0
HUQ010    : fractional values = 0
HUQ020    : fractional values = 0
HUQ030    : fractional values = 0
HUQ040    : fractional values = 0
HUQ050    : fractional values = 0
HUQ060    : fractional values = 0
HUQ071    : fractional values = 0
HUD080    : fractional values = 0
HUQ090    : fractional values = 0
PASS: All HUQ_D values are whole-number identifiers/codes.

--- SEQN CHECK ---
Missing: 0
Duplicates: 0
Minimum: 31127.0
Maximum: 41474.0
PASS: SEQN validated.

--- CDC 